In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [6]:
from pyspark.sql import functions as F

# Bronze dan qayta o'qish
df_bronze = spark.read.table(
    "`yellow_tripdata_2025-01_table`"
)

# To'liq tozalash
df_silver2 = df_bronze.filter(
    (F.col("fare_amount") > 0) &
    (F.col("trip_distance") > 0) &
    (F.col("passenger_count") > 0) &
    (F.col("passenger_count").isNotNull()) &
    (F.col("PULocationID").isNotNull()) &
    (F.col("DOLocationID").isNotNull()) &
    (F.col("tpep_pickup_datetime").isNotNull()) &
    (F.col("tpep_dropoff_datetime").isNotNull())
).withColumn("trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - 
     F.unix_timestamp("tpep_pickup_datetime")) / 60
).withColumn("pickup_hour", F.hour("tpep_pickup_datetime")
).withColumn("pickup_month", F.month("tpep_pickup_datetime")
).withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime")
).filter(F.col("trip_duration_minutes") > 0)

print("Qatorlar soni:", df_silver2.count())

StatementMeta(, e4d6af1a-a33c-488f-bd9a-0474d2accf85, 8, Finished, Available, Finished, False)

Qatorlar soni: 2815614


In [9]:
df_silver2.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_silver_clean")

print("✅ Silver ga saqlandi!")

StatementMeta(, e4d6af1a-a33c-488f-bd9a-0474d2accf85, 11, Finished, Available, Finished, False)

✅ Silver ga saqlandi!
